In [1]:
import h5py
import numpy as np
file_path = "/home/leiyang/Simulation_H5_for_Delay_Electron/20260818/muon/events_muon.h5"
# file_path = "/home/leiyang/Simulation_H5_for_Delay_Electron/20260818/gamma/events_gamma_OFF_SIDE_Y_OFF.h5"
# file_path = "/home/leiyang/Simulation_H5_for_Delay_Electron/Simulation_H5_for_Delay_Electron/events_muon_30M.h5/events_muon.h5"
# file_path = "/home/leiyang/Simulation_H5_for_Delay_Electron/Simulation_H5_for_Delay_Electron/events_Gamma/events_gamma_OFF_SIDE_Y_OFF.h5"
def print_structure(f):
    def visitor(name, obj):
        typ = "Group" if isinstance(obj, h5py.Group) else "Dataset"
        shape = getattr(obj, "shape", None)
        dtype = getattr(obj, "dtype", None)
        attrs = dict(obj.attrs) if len(obj.attrs) else {}
        print(f"{name}  ({typ})  shape={shape} dtype={dtype} attrs={attrs}")
    f.visititems(visitor)

with h5py.File(file_path, 'r') as f:
    print("Top keys:", list(f.keys()))
    # 打印完整结构，快速定位 cluster
    print_structure(f)

    # 假设存在名为 "cluster" 的 group 或 dataset
    if "clusters" in f:
        obj = f["clusters"]
        if isinstance(obj, h5py.Group):
            print("clusters 是 Group，包含：", list(obj.keys()))
            for name, ds in obj.items():
                if isinstance(ds, h5py.Dataset):
                    data = ds[:]            # 读取全部数据到 numpy
                    print(f"读取 dataset {name}: shape={data.shape}, dtype={data.dtype}")
                    # 如果是结构化数组：
                    if data.dtype.names:
                        print("字段：", data.dtype.names)
        else:  # 直接是 Dataset
            data = obj[:]
            print("clusters dataset shape:", data.shape, "dtype:", data.dtype)
            if data.dtype.names:
                print("字段：", data.dtype.names)
            # 示例访问字段
            # print(data['some_field'][:10])
    else:
        print("未找到名为 'clusters' 的对象，请检查键名（大小写敏感）。")

Top keys: ['clusters']
clusters  (Dataset)  shape=(8470939,) dtype=[('runId', '<u4'), ('eventId', '<u4'), ('energy', '<f8'), ('nr', '<f8'), ('er', '<f8'), ('er_x', '<f8'), ('er_y', '<f8'), ('er_z', '<f8'), ('er_t', '<f8'), ('er_parentId', '<u4'), ('er_trackId', '<u4'), ('er_type', 'O'), ('er_parentType', 'O'), ('er_creatorProcess', 'O'), ('er_depositionProcess', 'O'), ('er_volume', 'O'), ('nr_x', '<f8'), ('nr_y', '<f8'), ('nr_z', '<f8'), ('nr_t', '<f8'), ('nr_parentId', '<u4'), ('nr_trackId', '<u4'), ('nr_type', 'O'), ('nr_parentType', 'O'), ('nr_creatorProcess', 'O'), ('nr_depositionProcess', 'O'), ('nr_volume', 'O')] attrs={}
clusters dataset shape: (8470939,) dtype: [('runId', '<u4'), ('eventId', '<u4'), ('energy', '<f8'), ('nr', '<f8'), ('er', '<f8'), ('er_x', '<f8'), ('er_y', '<f8'), ('er_z', '<f8'), ('er_t', '<f8'), ('er_parentId', '<u4'), ('er_trackId', '<u4'), ('er_type', 'O'), ('er_parentType', 'O'), ('er_creatorProcess', 'O'), ('er_depositionProcess', 'O'), ('er_volume', 'O')

In [2]:
# ============================================================
# 基于 data 直接处理并保存（不依赖 muon_track / muon_track_sorted）
# 按 data['runId'] 每 RUNS_PER_FILE 个 run 合并为 1 个数组，分别保存
# 每个数组内 eventId 从 0 开始独立编号（同一 (run,event) 共享同一编号）
# ============================================================
import os
import numpy as np

PREFIX        = "muon_events"
SAVE_DIR      = "/home/leiyang/TPC_DE_SIm-test/muon_track/muon_events"
RUNS_PER_FILE = 30
START_IDX     = 1
os.makedirs(SAVE_DIR, exist_ok=True)

out_dtype = np.dtype([
    ('eventId', '<i8'), ('energy', '<f8'), ('xd', '<f8'),
    ('yd', '<f8'), ('zd', '<f8'), ('td', '<f8'),
])

# ---- 字段转换（物理量）----
run_ids = data['runId'].astype('<i8')
ev_ids  = data['eventId'].astype('<i8')
en      = data['energy'].astype('<f8')
xd      = data['er_x'].astype('<f8') + 150.0
yd      = data['er_y'].astype('<f8')
zd      = data['er_z'].astype('<f8')
td      = data['er_t'].astype('<f8')

# ---- RELICS5 选择 mask ----
mask2 = (zd > -520) & (zd < -370)
mask3 = np.sqrt(xd**2 + yd**2) < 82
sel   = mask2 & mask3

# ---- 被选择数据 ----
n_sel  = int(np.sum(sel))
s_run  = run_ids[sel]
s_arr  = np.empty(n_sel, dtype=out_dtype)
s_arr['energy'] = en[sel]
s_arr['xd'] = xd[sel]
s_arr['yd'] = yd[sel]
s_arr['zd'] = zd[sel]
s_arr['td'] = td[sel]
# 组合键：同一 (run, event) 唯一，用于排序与独立编号
s_key = s_run * 10000000 + ev_ids[sel]

# ---- 按 eventId 全局排序（使输出有序）----
order  = np.argsort(s_key)
s_key  = s_key[order]
s_run  = s_run[order]
s_arr  = s_arr[order]

# ---- 每 RUNS_PER_FILE 个 run 一组 ----
unique_run = np.unique(s_run)
run_no     = np.searchsorted(unique_run, s_run)   # 每个事件第几个 run
group_no   = run_no // RUNS_PER_FILE
n_groups   = int(group_no.max()) + 1
print(f"共 {len(unique_run)} 个 run，拆分为 {n_groups} 个文件（每文件 <= {RUNS_PER_FILE} run）")

# ---- 逐组保存，组内 eventId 从 0 独立编号 ----
for g in range(n_groups):
    idx = np.where(group_no == g)[0]
    part = s_arr[idx].copy()
    part['eventId'] = np.unique(s_key[idx], return_inverse=True)[1]
    n_run = len(np.unique(s_run[idx]))
    fname = f"{PREFIX}.{START_IDX + g}.npy"
    np.save(os.path.join(SAVE_DIR, fname), part)
    print(f"保存 {fname}: shape={part.shape}, 包含 run 数={n_run}")

共 300 个 run，拆分为 10 个文件（每文件 <= 30 run）
保存 muon_events.1.npy: shape=(549931,), 包含 run 数=30
保存 muon_events.2.npy: shape=(868069,), 包含 run 数=30
保存 muon_events.3.npy: shape=(658264,), 包含 run 数=30
保存 muon_events.4.npy: shape=(742948,), 包含 run 数=30
保存 muon_events.5.npy: shape=(744270,), 包含 run 数=30
保存 muon_events.6.npy: shape=(450576,), 包含 run 数=30
保存 muon_events.7.npy: shape=(481843,), 包含 run 数=30
保存 muon_events.8.npy: shape=(599810,), 包含 run 数=30
保存 muon_events.9.npy: shape=(764894,), 包含 run 数=30
保存 muon_events.10.npy: shape=(484952,), 包含 run 数=30


In [90]:
data

array([(  1, 37387526, 921.10146712, 0., 921.10146712, -107.13963478,  48.94455304, -525.28130934, 1.79762218e-08, 1, 0, b'e-', b'gamma', b'compt', b'eIoni', b'XenonDetector', nan, nan, nan, nan, 0, 0, b'unknown', b'unknown', b'unknown', b'unknown', b'unknown'),
       (  1, 37387526, 350.82734553, 0., 350.82734553, -101.30883901,  48.74428688, -529.18439352, 1.79962055e-08, 1, 0, b'e-', b'gamma', b'compt', b'eIoni', b'XenonDetector', nan, nan, nan, nan, 0, 0, b'unknown', b'unknown', b'unknown', b'unknown', b'unknown'),
       (  1, 37387526, 217.75157871, 0., 217.75157871, -100.60157917,  48.46844522, -520.5140634 , 1.80241938e-08, 1, 0, b'e-', b'gamma', b'phot', b'eIoni', b'XenonDetector', nan, nan, nan, nan, 0, 0, b'unknown', b'unknown', b'unknown', b'unknown', b'unknown'),
       ...,
       (300, 89366664,  25.98103246, 0.,  25.98103246, -218.04993595,  -8.08447278, -445.89421549, 1.77374055e-08, 1, 0, b'e-', b'gamma', b'compt', b'eIoni', b'XenonDetector', nan, nan, nan, nan, 0, 0

In [91]:
# target_dtype = np.dtype([
#     ('eventId', '<i4'),
#     ('energy', '<f8'),
#     ('xd', '<f8'),
#     ('yd', '<f8'),
#     ('zd', '<f8'),
#     ('td', '<f8')
# ])

# muon_track = np.empty(len(data), dtype=target_dtype)
# muon_track["eventId"] = data['eventId'].astype("<i4") if data['eventId'] is not None else np.arange(len(data))
# muon_track["energy"] = data['energy'].astype("<f8") if data['energy'] is not None else 0
# muon_track["xd"] = (data['er_x']+150).astype("<f8") if data['er_x'] is not None else 0
# muon_track["yd"] = data['er_y'].astype("<f8") if data['er_y'] is not None else 0
# muon_track["zd"] = data['er_z'].astype("<f8") if data['er_z'] is not None else 0
# muon_track["td"] = data['er_t'].astype("<f8") if data['er_t'] is not None else 0


In [92]:
import numpy as np

# 定义目标数据类型
target_dtype = np.dtype([
    ('eventId', '<i8'),  # 使用int64以容纳更大的值
    ('energy', '<f8'),
    ('xd', '<f8'),
    ('yd', '<f8'),
    ('zd', '<f8'),
    ('td', '<f8')
])

muon_track = np.empty(len(data), dtype=target_dtype)

# 组合runId和eventId生成唯一ID
# 方法A: 位移组合（假设runId和eventId都小于2^32）
run_id = data['runId'].astype('<i8')
event_id = data['eventId'].astype('<i8')
combined_id = run_id * 1e8 +  event_id  # 将runId放在高位，eventId放在低位

muon_track["eventId"] = combined_id

# 其他字段赋值
muon_track["energy"] = data['energy'].astype("<f8") if data['energy'] is not None else 0
muon_track["xd"] = (data['er_x'] + 150).astype("<f8") if data['er_x'] is not None else 0
muon_track["yd"] = data['er_y'].astype("<f8") if data['er_y'] is not None else 0
muon_track["zd"] = data['er_z'].astype("<f8") if data['er_z'] is not None else 0
muon_track["td"] = data['er_t'].astype("<f8") if data['er_t'] is not None else 0

In [93]:
print(min(muon_track['eventId']), max(muon_track['eventId']))
print(max(muon_track['zd']), min(muon_track['zd']))
print(max(muon_track['xd']), min(muon_track['xd']))
print(max(muon_track['yd']), min(muon_track['yd']))

137387526 30098740944
-370.02401354724753 -548.8426531219903
81.87390381894674 -82.44515388794184
82.1921797402944 -81.83235374638483


In [94]:
# RELICS5 mask

mask2 = (
    (muon_track['zd'] > -520) &
    (muon_track['zd'] < -370) 
)
mask3 = (np.sqrt(muon_track['xd']**2+muon_track['yd']**2)<82)

sel_muon_track = muon_track[mask2&mask3]

# 重新赋值
sorted_indices = np.argsort(sel_muon_track['eventId'])  
muon_track_sorted = sel_muon_track[sorted_indices]

muon_track_sorted['eventId'] = np.unique(muon_track_sorted['eventId'], return_inverse=True)[1]

In [95]:
muon_track_sorted['eventId']

array([   0,    0,    0, ..., 1476, 1476, 1477])

In [96]:
# np.save('/home/leiyang/TPC_DE_SIm-test/muon_track/muon_events/muon_events.1.npy', muon_track_sorted)
np.save('/home/leiyang/TPC_DE_SIm-test/muon_track/gamma_events/gamma_events.5.npy', muon_track_sorted)